In [ ]:
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/drive/MyDrive/row_total_data.csv',encoding='cp949')
df.head()

df.isna().sum()
df = df.fillna(df.mean())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.font_manager as fm  # 폰트 설정을 위한 라이브러리

# 사용자 정의 폰트 경로 설정
font_path = '/content/drive/MyDrive/MALGUN.TTF'

# 폰트를 matplotlib에 추가하고, 이를 사용하도록 설정
font_prop = fm.FontProperties(fname=font_path)
fm.fontManager.addfont(font_path)  # 폰트 추가
plt.rc('font', family=font_prop.get_name())  # 폰트 적용
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

# 사용할 온도 컬럼 선택
temperature_columns = [
     '소입로 온도 1 Zone', '소입로 온도 2 Zone', '소입로 온도 3 Zone', '소입로 온도 4 Zone'
]

# 상관 행렬 계산
correlation_matrix = df[temperature_columns].corr()

# 히트맵 그리기
plt.figure(figsize=(10, 8))  # 히트맵의 크기 설정
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5,
            annot_kws={"size": 12})  # 주석(숫자)의 폰트 크기를 12로 설정

# x축, y축 레이블 설정
ax = plt.gca()  # 현재 subplot의 축을 가져옴

# x축 레이블에서 'Zone'을 제외한 한글 폰트만 키우기
xticks_labels = [label.get_text().replace(' Zone', '') for label in ax.get_xticklabels()]
ax.set_xticklabels(xticks_labels, fontproperties=font_prop, fontsize=15)

# y축 레이블에서 'Zone'을 제외한 한글 폰트만 키우기
yticks_labels = [label.get_text().replace(' Zone', '') for label in ax.get_yticklabels()]
ax.set_yticklabels(yticks_labels, fontproperties=font_prop, fontsize=15)

# 폰트 속성을 제목에 직접 적용하고, 제목의 폰트 크기를 키움
plt.title("Temperature Correlation Matrix", fontproperties=font_prop, fontsize=20)

plt.show()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

# 사용할 온도 컬럼 선택
temperature_columns = [
    '소입로 온도 1 Zone', '소입로 온도 2 Zone', '소입로 온도 3 Zone', '소입로 온도 4 Zone'
]

# 데이터 로드
temperature_data = df[temperature_columns].values

# 데이터 전처리 (정규화)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(temperature_data)

# 학습 데이터와 테스트 데이터 분리 (80% 학습, 20% 테스트)
X_train, X_test = train_test_split(scaled_data, test_size=0.2, shuffle=False)

# One-Class SVM 모델 생성 (nu는 이상치 비율, kernel은 RBF 사용)
model = OneClassSVM(nu=0.1, kernel="rbf", gamma='scale')

# 정상 데이터로 모델 학습
model.fit(X_train)

# 테스트 데이터로 예측
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# -----------------------------
# 실제 평균값 기반 ±5°C 이상치 라벨링
# -----------------------------

# 각 Zone의 평균값 계산 (정규화 전 원래 데이터에서)
mean_temperatures = np.mean(temperature_data, axis=0)

# ±5°C 기준으로 오차 계산 (5°C 이상 차이가 나는 값은 이상치로 간주)
threshold_abs = 5  # 5°C 기준
y_true_test = np.zeros_like(X_test)  # 이상치를 라벨링할 배열 (정상: 1, 이상: -1)

# 평균과 ±5°C 이상 차이가 나는 경우 이상치로 라벨링
for i in range(X_test.shape[0]):
    for j in range(len(temperature_columns)):
        if abs(temperature_data[i + len(X_train), j] - mean_temperatures[j]) > threshold_abs:
            y_true_test[i, j] = -1  # 이상치로 라벨링

# 모든 Zone에 대해 하나라도 이상치가 있으면 -1로 설정, 그렇지 않으면 1
y_true_test = np.where(np.any(y_true_test == -1, axis=1), -1, 1)

# -----------------------------
# 모델 평가
# -----------------------------
def evaluate_model(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=-1)
    recall = recall_score(y_true, y_pred, pos_label=-1)
    f1 = f1_score(y_true, y_pred, pos_label=-1)
    return accuracy, precision, recall, f1

# 평가 지표 계산
test_accuracy, test_precision, test_recall, test_f1 = evaluate_model(y_true_test, y_pred_test)

# 결과 출력
print(f"Test Accuracy: {test_accuracy:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1 Score: {test_f1:.4f}")

# 각 Zone의 평균 출력
for i, column in enumerate(temperature_columns):
    print(f"Average temperature in {column}: {mean_temperatures[i]:.2f}°C")

# 전체 평균 계산
overall_mean_temperature = np.mean(mean_temperatures)
print(f"\nOverall average temperature across all zones: {overall_mean_temperature:.2f}°C")

import numpy as np
import matplotlib.pyplot as plt
from math import pi

# 평가지표
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

# 테스트 결과 (사전으로 정의)
test_scores_dict = {'Accuracy': 0.8856, 'Precision': 0.6116, 'Recall': 0.7729, 'F1 Score': 0.6828}

# 사전에서 점수만 추출하여 리스트로 변환
test_scores = list(test_scores_dict.values())

# 레이다 차트 준비
labels = metrics
num_vars = len(labels)

# 각 지표에 대한 각도 계산
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # 마지막 점을 첫 번째 점과 연결

# 데이터에 마지막 값을 첫 번째 값으로 추가 (차트를 닫기 위함)
test_scores += test_scores[:1]

# 레이다 차트 그리기
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# 두 번째 데이터: Test Scores
ax.fill(angles, test_scores, color='green', alpha=0.25)
ax.plot(angles, test_scores, color='green', linewidth=2, label='Test Scores')

# 레이블 설정
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)

# Y축 범위 설정 (0에서 1로 고정)
ax.set_ylim(0, 1)

# 레전드와 타이틀 추가
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.title('Test Evaluation Metrics')

plt.show()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

# 사용할 온도 컬럼 선택
temperature_columns = [
   '건조로 온도 1 Zone', '건조로 온도 2 Zone'
]

# 데이터 로드
temperature_data = df[temperature_columns].values

# 데이터 전처리 (정규화)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(temperature_data)

# 학습 데이터와 테스트 데이터 분리 (80% 학습, 20% 테스트)
X_train, X_test = train_test_split(scaled_data, test_size=0.2, shuffle=False)

# One-Class SVM 모델 생성 (nu는 이상치 비율, kernel은 RBF 사용)
model = OneClassSVM(nu=0.1, kernel="rbf", gamma='scale')

# 정상 데이터로 모델 학습
model.fit(X_train)

# 테스트 데이터로 예측
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# -----------------------------
# 실제 평균값 기반 ±5°C 이상치 라벨링
# -----------------------------

# 각 Zone의 평균값 계산 (정규화 전 원래 데이터에서)
mean_temperatures = np.mean(temperature_data, axis=0)

# ±5°C 기준으로 오차 계산 (5°C 이상 차이가 나는 값은 이상치로 간주)
threshold_abs = 5  # 5°C 기준
y_true_test = np.zeros_like(X_test)  # 이상치를 라벨링할 배열 (정상: 1, 이상: -1)

# 평균과 ±5°C 이상 차이가 나는 경우 이상치로 라벨링
for i in range(X_test.shape[0]):
    for j in range(len(temperature_columns)):
        if abs(temperature_data[i + len(X_train), j] - mean_temperatures[j]) > threshold_abs:
            y_true_test[i, j] = -1  # 이상치로 라벨링

# 모든 Zone에 대해 하나라도 이상치가 있으면 -1로 설정, 그렇지 않으면 1
y_true_test = np.where(np.any(y_true_test == -1, axis=1), -1, 1)

# -----------------------------
# 모델 평가
# -----------------------------
def evaluate_model(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=-1)
    recall = recall_score(y_true, y_pred, pos_label=-1)
    f1 = f1_score(y_true, y_pred, pos_label=-1)
    return accuracy, precision, recall, f1

# 평가 지표 계산
test_accuracy, test_precision, test_recall, test_f1 = evaluate_model(y_true_test, y_pred_test)

# 결과 출력
print(f"Test Accuracy: {test_accuracy:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1 Score: {test_f1:.4f}")

# 각 Zone의 평균 출력
for i, column in enumerate(temperature_columns):
    print(f"Average temperature in {column}: {mean_temperatures[i]:.2f}°C")

# 전체 평균 계산
overall_mean_temperature = np.mean(mean_temperatures)
print(f"\nOverall average temperature across all zones: {overall_mean_temperature:.2f}°C")

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
train_scores = [train_accuracy, train_precision, train_recall, train_f1]
test_scores = [test_accuracy, test_precision, test_recall, test_f1]

# 레이다 차트 준비
labels = metrics
num_vars = len(labels)

# 각 지표에 대한 각도 계산
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # 마지막 점을 첫 번째 점과 연결

# 레이다 차트 그리기
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# 첫 번째 데이터: Train Scores
train_scores += train_scores[:1]  # 마지막 점을 첫 번째 점과 연결
ax.fill(angles, train_scores, color='blue', alpha=0.25)
ax.plot(angles, train_scores, color='blue', linewidth=2, label='Train Scores')

# 두 번째 데이터: Test Scores
test_scores += test_scores[:1]  # 마지막 점을 첫 번째 점과 연결
ax.fill(angles, test_scores, color='green', alpha=0.25)
ax.plot(angles, test_scores, color='green', linewidth=2, label='Test Scores')

# 레이블 설정
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)

# Y축 범위 설정 (0에서 1로 고정)
ax.set_ylim(0, 1)

# 레전드와 타이틀 추가
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.title('Test Evaluation Metrics')

plt.show()

temperature_columns = [
    '솔트 컨베이어 온도 1 Zone', '솔트 컨베이어 온도 2 Zone'
]

# 데이터 로드
temperature_data = df[temperature_columns].values

# 데이터 전처리 (정규화)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(temperature_data)

# 학습 데이터와 테스트 데이터 분리 (80% 학습, 20% 테스트)
X_train, X_test = train_test_split(scaled_data, test_size=0.2, shuffle=False)

# One-Class SVM 모델 생성 (nu는 이상치 비율, kernel은 RBF 사용)
model = OneClassSVM(nu=0.1, kernel="rbf", gamma='scale')

# 정상 데이터로 모델 학습
model.fit(X_train)

# 테스트 데이터로 예측
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# -----------------------------
# 실제 평균값 기반 ±5°C 이상치 라벨링
# -----------------------------

# 각 Zone의 평균값 계산 (정규화 전 원래 데이터에서)
mean_temperatures = np.mean(temperature_data, axis=0)

# ±5°C 기준으로 오차 계산 (5°C 이상 차이가 나는 값은 이상치로 간주)
threshold_abs = 5  # 5°C 기준
y_true_test = np.zeros_like(X_test)  # 이상치를 라벨링할 배열 (정상: 1, 이상: -1)

# 평균과 ±5°C 이상 차이가 나는 경우 이상치로 라벨링
for i in range(X_test.shape[0]):
    for j in range(len(temperature_columns)):
        if abs(temperature_data[i + len(X_train), j] - mean_temperatures[j]) > threshold_abs:
            y_true_test[i, j] = -1  # 이상치로 라벨링

# 모든 Zone에 대해 하나라도 이상치가 있으면 -1로 설정, 그렇지 않으면 1
y_true_test = np.where(np.any(y_true_test == -1, axis=1), -1, 1)

# -----------------------------
# 모델 평가
# -----------------------------
def evaluate_model(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=-1)
    recall = recall_score(y_true, y_pred, pos_label=-1)
    f1 = f1_score(y_true, y_pred, pos_label=-1)
    return accuracy, precision, recall, f1

# 평가 지표 계산
test_accuracy, test_precision, test_recall, test_f1 = evaluate_model(y_true_test, y_pred_test)

# 결과 출력
print(f"Test Accuracy: {test_accuracy:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1 Score: {test_f1:.4f}")

# 각 Zone의 평균 출력
for i, column in enumerate(temperature_columns):
    print(f"Average temperature in {column}: {mean_temperatures[i]:.2f}°C")

# 전체 평균 계산
overall_mean_temperature = np.mean(mean_temperatures)
print(f"\nOverall average temperature across all zones: {overall_mean_temperature:.2f}°C")

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
train_scores = [train_accuracy, train_precision, train_recall, train_f1]
test_scores = [test_accuracy, test_precision, test_recall, test_f1]

# 레이다 차트 준비
labels = metrics
num_vars = len(labels)

# 각 지표에 대한 각도 계산
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # 마지막 점을 첫 번째 점과 연결

# 레이다 차트 그리기
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# 첫 번째 데이터: Train Scores
train_scores += train_scores[:1]  # 마지막 점을 첫 번째 점과 연결
ax.fill(angles, train_scores, color='blue', alpha=0.25)
ax.plot(angles, train_scores, color='blue', linewidth=2, label='Train Scores')

# 두 번째 데이터: Test Scores
test_scores += test_scores[:1]  # 마지막 점을 첫 번째 점과 연결
ax.fill(angles, test_scores, color='green', alpha=0.25)
ax.plot(angles, test_scores, color='green', linewidth=2, label='Test Scores')

# 레이블 설정
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)

# Y축 범위 설정 (0에서 1로 고정)
ax.set_ylim(0, 1)

# 레전드와 타이틀 추가
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.title('Test Evaluation Metrics')

plt.show()

temperature_columns = [
    '솔트조 온도 1 Zone', '솔트조 온도 2 Zone'
]

# 데이터 로드
temperature_data = df[temperature_columns].values

# 데이터 전처리 (정규화)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(temperature_data)

# 학습 데이터와 테스트 데이터 분리 (80% 학습, 20% 테스트)
X_train, X_test = train_test_split(scaled_data, test_size=0.2, shuffle=False)

# One-Class SVM 모델 생성 (nu는 이상치 비율, kernel은 RBF 사용)
model = OneClassSVM(nu=0.1, kernel="rbf", gamma='scale')

# 정상 데이터로 모델 학습
model.fit(X_train)

# 테스트 데이터로 예측
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# -----------------------------
# 실제 평균값 기반 ±5°C 이상치 라벨링
# -----------------------------

# 각 Zone의 평균값 계산 (정규화 전 원래 데이터에서)
mean_temperatures = np.mean(temperature_data, axis=0)

# ±5°C 기준으로 오차 계산 (5°C 이상 차이가 나는 값은 이상치로 간주)
threshold_abs = 5  # 5°C 기준
y_true_test = np.zeros_like(X_test)  # 이상치를 라벨링할 배열 (정상: 1, 이상: -1)

# 평균과 ±5°C 이상 차이가 나는 경우 이상치로 라벨링
for i in range(X_test.shape[0]):
    for j in range(len(temperature_columns)):
        if abs(temperature_data[i + len(X_train), j] - mean_temperatures[j]) > threshold_abs:
            y_true_test[i, j] = -1  # 이상치로 라벨링

# 모든 Zone에 대해 하나라도 이상치가 있으면 -1로 설정, 그렇지 않으면 1
y_true_test = np.where(np.any(y_true_test == -1, axis=1), -1, 1)

# -----------------------------
# 모델 평가
# -----------------------------
def evaluate_model(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=-1)
    recall = recall_score(y_true, y_pred, pos_label=-1)
    f1 = f1_score(y_true, y_pred, pos_label=-1)
    return accuracy, precision, recall, f1

# 평가 지표 계산
test_accuracy, test_precision, test_recall, test_f1 = evaluate_model(y_true_test, y_pred_test)

# 결과 출력
print(f"Test Accuracy: {test_accuracy:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1 Score: {test_f1:.4f}")

# 각 Zone의 평균 출력
for i, column in enumerate(temperature_columns):
    print(f"Average temperature in {column}: {mean_temperatures[i]:.2f}°C")

# 전체 평균 계산
overall_mean_temperature = np.mean(mean_temperatures)
print(f"\nOverall average temperature across all zones: {overall_mean_temperature:.2f}°C")

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
train_scores = [train_accuracy, train_precision, train_recall, train_f1]
test_scores = [test_accuracy, test_precision, test_recall, test_f1]

# 레이다 차트 준비
labels = metrics
num_vars = len(labels)

# 각 지표에 대한 각도 계산
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # 마지막 점을 첫 번째 점과 연결

# 레이다 차트 그리기
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# 첫 번째 데이터: Train Scores
train_scores += train_scores[:1]  # 마지막 점을 첫 번째 점과 연결
ax.fill(angles, train_scores, color='blue', alpha=0.25)
ax.plot(angles, train_scores, color='blue', linewidth=2, label='Train Scores')

# 두 번째 데이터: Test Scores
test_scores += test_scores[:1]  # 마지막 점을 첫 번째 점과 연결
ax.fill(angles, test_scores, color='green', alpha=0.25)
ax.plot(angles, test_scores, color='green', linewidth=2, label='Test Scores')

# 레이블 설정
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)

# Y축 범위 설정 (0에서 1로 고정)
ax.set_ylim(0, 1)

# 레전드와 타이틀 추가
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.title('Test Evaluation Metrics')

plt.show()

#AE
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings(action='ignore')

df = pd.read_csv('/content/drive/MyDrive/row_total_data.csv',encoding='cp949')
df.head()

df.describe()
df.isnull().sum()
df = df.fillna(df.mean())
for col in df.columns:
 print( col,':', len(df[col].value_counts()))

df['이상치개수'] =0
for col in df.columns:
 thirdq, firstq = df[col].quantile(0.75), df[col].quantile(0.25)
 interquartilerange =1.5*(thirdq-firstq)
 outlierhigh, outlierlow = interquartilerange+thirdq, firstq-interquartilerange
 for i in df.index:
  if df[col][i] > outlierhigh or df[col][i] < outlierlow:
    df['이상치개수'][i] +=1
df.head()

df['이상치개수'].value_counts()

df['설비 이상신호'] =np.where(df['이상치개수'] < 9, 0, 1)
df['설비 이상신호'].value_counts()

normal = df.loc[df['설비 이상신호'] ==0]
abnormal = df.loc[df['설비 이상신호'] ==1]
print('정상 데이터:', len(normal))
print('설비이상 데이터:', len(abnormal))

normal.drop(columns='이상치개수', inplace=True)
abnormal.drop(columns='이상치개수', inplace=True)

from sklearn.model_selection import train_test_split
train_Y, test_Y = train_test_split(normal, test_size =0.3, random_state =1)
print('학습 데이터셋 개수:', len(train_Y))
print('테스트 데이터셋 개수:', len(test_Y))

from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 스케일링
scaler = MinMaxScaler()

# 정상 학습데이터 스케일링
normal_train_scaled = scaler.fit_transform(train_Y.iloc[:, :-1])
X_normal_train = pd.DataFrame(data=normal_train_scaled,
                              index=train_Y.index,  # 인덱스를 train_Y 전체 데이터의 인덱스로 설정
                              columns=train_Y.columns[:-1])  # 컬럼을 마지막 y값 컬럼 제외

# 정상 학습데이터 y 값 설정
y_normal_train = train_Y.iloc[:, -1]

# 최종 정상 학습데이터
train_Y_normal = pd.concat([X_normal_train, y_normal_train], axis=1)

# 정상 테스트데이터 스케일링
normal_test_scaled = scaler.transform(test_Y.iloc[:, :-1])
X_normal_test = pd.DataFrame(data=normal_test_scaled,
                             index=test_Y.index,  # 인덱스를 test_Y 전체 데이터의 인덱스로 설정
                             columns=test_Y.columns[:-1])  # 컬럼을 마지막 y값 컬럼 제외

# 정상 테스트데이터 y 값 설정
y_normal_test = test_Y.iloc[:, -1]

# 최종 정상 테스트데이터
test_Y_normal = pd.concat([X_normal_test, y_normal_test], axis=1)

# 비정상 테스트 데이터 스케일링
abnormal_test_scaled = scaler.transform(abnormal.iloc[:, :-1])
X_abnormal_test = pd.DataFrame(data=abnormal_test_scaled,
                               index=abnormal.index,  # 인덱스를 abnormal 데이터의 인덱스로 설정
                               columns=abnormal.columns[:-1])  # 컬럼을 마지막 y값 컬럼 제외

# 비정상 테스트데이터 y 값 설정
y_abnormal_test = abnormal.iloc[:, -1]

# 최종 비정상 테스트데이터
abnormal_data = pd.concat([X_abnormal_test, y_abnormal_test], axis=1)

import tensorflow as tf
tf.random.set_seed(1)
from keras.layers import Dense
from keras.models import Sequential

# 모델 구축
model_encoder = Sequential([
 Dense(15, activation="relu"),
 Dense(10, activation="relu"),
 Dense(5, activation="relu")
])
model_decoder = Sequential([
 Dense(10, activation="relu", input_shape=[5]),
 Dense(15, activation="relu"),
 Dense(X_normal_train.shape[1], activation="relu")
])
AE_model =Sequential([model_encoder, model_decoder])

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Model Checkpoint 경로 수정 (accuracy를 포함하도록 변경)
model_path = './model/{epoch:02d}-{val_accuracy:.4f}.keras'

# 모델 학습, 콜백 설정
history = AE_model.fit(
    X_normal_train, X_normal_train,
    batch_size=32,
    epochs=50,
    validation_split=0.2,
    callbacks=[
        EarlyStopping(monitor="val_accuracy", patience=10, mode="max"),
        ModelCheckpoint(filepath=model_path, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
    ]
)

#LSTM-AE

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings(action='ignore')

df = pd.read_csv('/content/drive/MyDrive/row_total_data.csv',encoding='cp949')
df.head()

df = df.fillna(df.mean())

df['이상치개수'] =0
for col in df.columns:
 thirdq, firstq = df[col].quantile(0.75), df[col].quantile(0.25)
 interquartilerange =1.5*(thirdq-firstq)
 outlierhigh, outlierlow = interquartilerange+thirdq, firstq-interquartilerange
 for i in df.index:
  if df[col][i] > outlierhigh or df[col][i] < outlierlow:
    df['이상치개수'][i] +=1
df.head()

df['이상치개수'].value_counts()

df['설비 이상신호'] =np.where(df['이상치개수'] < 9, 0, 1)
df['설비 이상신호'].value_counts()

normal = df.loc[df['설비 이상신호'] ==0]
abnormal = df.loc[df['설비 이상신호'] ==1]
print('정상 데이터:', len(normal))
print('설비이상 데이터:', len(abnormal))

normal.drop(columns='이상치개수', inplace=True)
abnormal.drop(columns='이상치개수', inplace=True)

from sklearn.model_selection import train_test_split
train_Y, test_Y = train_test_split(normal, test_size =0.3, random_state =1)
print('학습 데이터셋 개수:', len(train_Y))
print('테스트 데이터셋 개수:', len(test_Y))

from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 스케일링
scaler = MinMaxScaler()

# 정상 학습데이터 스케일링
normal_train_scaled = scaler.fit_transform(train_Y.iloc[:, :-1])
X_normal_train = pd.DataFrame(data=normal_train_scaled,
                              index=train_Y.index,  # 인덱스를 train_Y 전체 데이터의 인덱스로 설정
                              columns=train_Y.columns[:-1])  # 컬럼을 마지막 y값 컬럼 제외

# 정상 학습데이터 y 값 설정
y_normal_train = train_Y.iloc[:, -1]

# 최종 정상 학습데이터
train_Y_normal = pd.concat([X_normal_train, y_normal_train], axis=1)

# 정상 테스트데이터 스케일링
normal_test_scaled = scaler.transform(test_Y.iloc[:, :-1])
X_normal_test = pd.DataFrame(data=normal_test_scaled,
                             index=test_Y.index,  # 인덱스를 test_Y 전체 데이터의 인덱스로 설정
                             columns=test_Y.columns[:-1])  # 컬럼을 마지막 y값 컬럼 제외

# 정상 테스트데이터 y 값 설정
y_normal_test = test_Y.iloc[:, -1]

# 최종 정상 테스트데이터
test_Y_normal = pd.concat([X_normal_test, y_normal_test], axis=1)

# 비정상 테스트 데이터 스케일링
abnormal_test_scaled = scaler.transform(abnormal.iloc[:, :-1])
X_abnormal_test = pd.DataFrame(data=abnormal_test_scaled,
                               index=abnormal.index,  # 인덱스를 abnormal 데이터의 인덱스로 설정
                               columns=abnormal.columns[:-1])  # 컬럼을 마지막 y값 컬럼 제외

# 비정상 테스트데이터 y 값 설정
y_abnormal_test = abnormal.iloc[:, -1]

# 최종 비정상 테스트데이터
abnormal_data = pd.concat([X_abnormal_test, y_abnormal_test], axis=1)

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt

# 사용자 정의 정확도 메트릭 (reconstruction accuracy)
def reconstruction_accuracy(y_true, y_pred):
    # 임계값을 사용한 재구성 정확도
    threshold = 0.55  # 재구성 차이의 허용 범위 임계값
    diff = K.abs(y_true - y_pred)
    correct = K.mean(K.less(diff, threshold), axis=-1)
    return correct

# LSTM Autoencoder 모델 구성
timesteps = 1  # 시퀀스 길이
features = X_normal_train.shape[1]  # 입력 특성 수

# 인코더 (차원 축소)
model_encoder = Sequential([
    LSTM(64, activation="relu", input_shape=(timesteps, features), return_sequences=False),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(5, activation="relu")  # 잠재 공간
])

# 디코더 (차원 복원)
model_decoder = Sequential([
    Dense(16, activation="relu", input_shape=[5]),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(64, activation="relu"),
    Dense(features, activation="sigmoid")  # 입력과 동일한 차원으로 복원
])

# LSTM Autoencoder 전체 모델
AE_model = Sequential([model_encoder, model_decoder])

# 모델 컴파일 (손실: MSE, 메트릭: 사용자 정의 정확도)
AE_model.compile(optimizer='adam', loss='mse', metrics=[reconstruction_accuracy])

# LSTM 모델에 맞게 3차원으로 데이터 변환 (samples, timesteps, features)
X_train_lstm = np.array(X_normal_train).reshape((X_normal_train.shape[0], timesteps, features))

# 모델 훈련
history = AE_model.fit(
    X_train_lstm, X_train_lstm,  # 입력과 출력이 동일 (Autoencoder)
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=10, mode="min"),
        ModelCheckpoint(filepath='best_lstm_autoencoder.keras', monitor='val_loss', save_best_only=True)
    ]
)

# 모델 평가
loss, accuracy = AE_model.evaluate(X_train_lstm, X_train_lstm)
print(f"재구성 정확도: {accuracy}")

# 학습 과정 시각화
plt.plot(history.history['reconstruction_accuracy'], label='Train Accuracy')
plt.plot(history.history['val_reconstruction_accuracy'], label='Validation Accuracy')
plt.title('LSTM Autoencoder Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.show()

# 손실 그래프 시각화
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('LSTM Autoencoder Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.show()